In [3]:
#Project to build 
#An end-to-end deep learning system that can predict where someone is standing inside a building using only the WiFi signals their device detects.


#Project Member: Shaeekh Al Jahan
#Declarion of AI: Took help from GPT to generate graphs and format descriptions

#_____Phase 2______
#Dataset Description: The UJIIndoorLoc dataset contains around ~20,000 real fingerprint samples collected across 3 university buildings in Spain.
#Each sample records signal strength (RSSI) from 520 access points, along with the exact longitude, latitude, and floor where the measurement was taken.

#Detailed Descriptions of this phase
#2.1 — Loading Phase 1 outputs. These include saved tensors and phase1_config.json directly from the archive path.
#2.2 — Model architecture. The IndoorLocMLP class with two hidden layers (512 → 256), BatchNorm, ReLU, Dropout(0.3). Fully documented with a reasoning table for every design choice.
#2.3 — Inspect the model. Prints the full architecture and total parameter count (~410k trainable parameters).
#2.4 — Forward pass sanity check. Runs one batch through the model to confirm input/output shapes are correct before beginning of training
#2.5 — Loss & optimiser preview. Defines MSELoss, Adam, and a ReduceLROnPlateau scheduler — all ready to be picked up in Phase 3.
#2.6 — Save for Phase 3. Saves model_init.pt (untrained weights) and phase2_config.json so Phase 3 can load everything cleanly.

# UJIIndoorLoc — Phase 2: Neural Network Design

**Goal:** Define the PyTorch model architecture, verify it loads Phase 1 outputs correctly, run a forward pass sanity check, and inspect the model.

**Inputs (from Phase 1):**
- `X_train.pt`, `y_train.pt` — training tensors
- `X_val.pt`, `y_val.pt` — validation tensors
- `phase1_config.json` — dataset constants

**Deliverable:** A fully documented model class ready for Phase 3 training.

---
## 2.0  Imports

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')

PyTorch version : 2.9.1+cpu
CUDA available  : False
Using device    : cpu


---
## 2.1  Load Phase 1 Outputs

In [5]:
#Paths for saved outputs from phase 1
BASE_PATH = '.'

X_train = torch.load(f'{BASE_PATH}\\X_train.pt')
y_train = torch.load(f'{BASE_PATH}\\y_train.pt')
X_val   = torch.load(f'{BASE_PATH}\\X_val.pt')
y_val   = torch.load(f'{BASE_PATH}\\y_val.pt')

with open(f'{BASE_PATH}\\phase1_config.json') as f:
    config = json.load(f)

INPUT_DIM = X_train.shape[1]
print(f'X_train shape : {X_train.shape}')   # (19937, 520)
print(f'y_train shape : {y_train.shape}')   # (19937, 2)
print(f'X_val shape   : {X_val.shape}')     # (1111, 520)
print(f'y_val shape   : {y_val.shape}')     # (1111, 2)
print(f'Input dim     : {INPUT_DIM}')

X_train shape : torch.Size([19937, 465])
y_train shape : torch.Size([19937, 2])
X_val shape   : torch.Size([1111, 465])
y_val shape   : torch.Size([1111, 2])
Input dim     : 465


In [6]:
#Reasons for having input dim 465 is some of the rows of 520 were dropped. The dropped ones had 100 [ missing of beacon reading] values only

---
## 2.2  Model Architecture

### Design decisions

| Choice | Reasoning |
|---|---|
| Two hidden layers (512 → 256) | Deep enough to learn non-linear fingerprint patterns; shallow enough to train fast on ~20k samples |
| BatchNorm1d after each Linear | Input is ~97% zeros (sparse); BatchNorm stabilises gradients and speeds convergence |
| ReLU activation | Standard, fast, avoids vanishing gradient for positive activations |
| Dropout(0.3) | Prevents the model from memorising individual fingerprints; improves generalisation |
| Output size = 2 | Predicts normalised (longitude, latitude) simultaneously |
| No activation on output | Regression output — values stay in [0, 1] because targets were MinMax-scaled |

```
Input  (520)
  │
  ├─ Linear(520 → 512)
  ├─ BatchNorm1d(512)
  ├─ ReLU
  ├─ Dropout(0.3)
  │
  ├─ Linear(512 → 256)
  ├─ BatchNorm1d(256)
  ├─ ReLU
  ├─ Dropout(0.3)
  │
  └─ Linear(256 → 2)   ← (x_norm, y_norm)
```

In [7]:
class IndoorLocMLP(nn.Module):
    """
    Two-hidden-layer MLP for WiFi fingerprint-based indoor positioning.

    Architecture
    ------------
    Input  (input_dim)  →  512  →  256  →  2 (x, y)
    Each hidden layer:  Linear → BatchNorm1d → ReLU → Dropout

    Parameters
    ----------
    input_dim : int
        Number of WiFi access points (520 for UJIIndoorLoc).
    hidden1 : int
        Neurons in first hidden layer (default 512).
    hidden2 : int
        Neurons in second hidden layer (default 256).
    dropout_p : float
        Dropout probability applied after each hidden layer (default 0.3).
    output_dim : int
        Number of output values — 2 for (longitude, latitude).
    """

    def __init__(
        self,
        input_dim: int  = 520,
        hidden1:   int  = 512,
        hidden2:   int  = 256,
        dropout_p: float = 0.3,
        output_dim: int = 2
    ):
        super(IndoorLocMLP, self).__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(p=dropout_p)
        )

        #  Block 2 
        self.block2 = nn.Sequential(
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(p=dropout_p)
        )


        self.output_layer = nn.Linear(hidden2, output_dim)

      
        self._init_weights()

    def _init_weights(self):
        """He (Kaiming) initialisation for ReLU networks."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor, shape (batch_size, input_dim)
            Normalised RSSI fingerprint vector.

        Returns
      
        torch.Tensor, shape (batch_size, 2)
            Predicted normalised (longitude, latitude).
        """
        x = self.block1(x)
        x = self.block2(x)
        return self.output_layer(x)

---
## 2.3  Instantiate & Inspect the Model

In [8]:
model = IndoorLocMLP(input_dim=INPUT_DIM).to(DEVICE)
print(model)

IndoorLocMLP(
  (block1): Sequential(
    (0): Linear(in_features=465, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (output_layer): Linear(in_features=256, out_features=2, bias=True)
)


In [9]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

Total parameters     : 371,970
Trainable parameters : 371,970


---
## 2.4  Forward Pass Sanity Check

Pass a single batch through the model to confirm shapes are correct before training.

In [10]:
from torch.utils.data import TensorDataset, DataLoader


BATCH_SIZE = 256

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.eval()
with torch.no_grad():
    sample_X, sample_y = next(iter(train_loader))
    sample_X = sample_X.to(DEVICE)
    pred = model(sample_X)

print(f'Input batch shape  : {sample_X.shape}')   # (256, 520)
print(f'Output batch shape : {pred.shape}')        # (256, 2)
print(f'Sample output (first 3 rows):')
print(pred[:3].cpu().numpy())
print('\n✅ Forward pass OK — shapes are correct.')

Input batch shape  : torch.Size([256, 465])
Output batch shape : torch.Size([256, 2])
Sample output (first 3 rows):
[[-0.01551761 -0.05301941]
 [-0.02470432  0.04485058]
 [-0.03075595  0.03466162]]

✅ Forward pass OK — shapes are correct.


---
## 2.5  Loss Function & Optimiser Preview

We define these here so Phase 3 can import them directly.

In [12]:

criterion = nn.MSELoss()


LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4

optimiser = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser,
    mode='min',
    factor=0.5,       # halve LR on plateau
    patience=5,       # wait 5 epochs before reducing
   # verbose=True
)

print('Loss      :', criterion)
print('Optimiser :', optimiser)
print('Scheduler : ReduceLROnPlateau (factor=0.5, patience=5)')

Loss      : MSELoss()
Optimiser : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)
Scheduler : ReduceLROnPlateau (factor=0.5, patience=5)


---
## 2.6  Save Model & Config for Phase 3

In [13]:
import pickle

torch.save(model.state_dict(), f'{BASE_PATH}\\model_init.pt')

#Save Phase 2 config
phase2_config = {
    'input_dim'     : INPUT_DIM,
    'hidden1'       : 512,
    'hidden2'       : 256,
    'dropout_p'     : 0.3,
    'output_dim'    : 2,
    'batch_size'    : BATCH_SIZE,
    'learning_rate' : LEARNING_RATE,
    'weight_decay'  : WEIGHT_DECAY,
    'device'        : str(DEVICE)
}

with open(f'{BASE_PATH}\\phase2_config.json', 'w') as f:
    json.dump(phase2_config, f, indent=2)

print('Saved: model_init.pt')
print('Saved: phase2_config.json')
print()
print('Phase 2 config summary:')
for k, v in phase2_config.items():
    print(f'  {k:<16}: {v}')

Saved: model_init.pt
Saved: phase2_config.json

Phase 2 config summary:
  input_dim       : 465
  hidden1         : 512
  hidden2         : 256
  dropout_p       : 0.3
  output_dim      : 2
  batch_size      : 256
  learning_rate   : 0.001
  weight_decay    : 0.0001
  device          : cpu


---
## 2.7  Phase 2 Summary

| Component | Choice | Reason |
|---|---|---|
| Architecture | MLP 520 → 512 → 256 → 2 | Proven effective for fingerprint regression |
| Activation | ReLU | Fast, no vanishing gradient for positive values |
| Normalisation | BatchNorm1d | Handles 97% sparse input; stabilises training |
| Regularisation | Dropout(0.3) + Weight decay(1e-4) | Prevents fingerprint memorisation |
| Weight init | Kaiming (He) normal | Optimal for ReLU networks |
| Loss | MSELoss | Standard for coordinate regression |
| Optimiser | Adam(lr=1e-3, wd=1e-4) | Adaptive LR; handles sparse gradients well |
| LR schedule | ReduceLROnPlateau | Automatically adapts when learning stalls |

### What's next — Phase 3
- Full training loop with early stopping
- Track train loss and val loss per epoch
- Plot learning curves
- Save best model checkpoint (`best_model.pt`)
- Experiment with batch size and dropout rate